# TRAIN+EVAL+VAL

In [2]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 36.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 14.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 10.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 90.3 MB/s  0:00:006m0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 89.3 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 20.8 MB/s  0:

In [ ]:
"""
Midm-2.0-Mini-Instruct + LoRA SFT 학습 스크립트 (전체 코드)

기능 요약
1) category.csv(code, name)로 train_with_json.csv 전체 코드값 치환
2) 8:1:1 (train / eval / val) 분할
3) LoRA 기반 SFT 학습 (SYSTEM_PROMPT는 역할/태스크만 정의)
4) 학습 중 tqdm 진행률 출력
5) 각 Epoch마다 eval 세트로 BLEU / ROUGE-L / METEOR / BERTScore / Embedding Cosine / WeightedScore 계산
6) 학습 완료 후 val 세트 평가
7) furniture.csv(goods_name) 기반으로 DESC와 유사한 가구 Top-K 추천 예시

전제 CSV 스키마
- train_with_json.csv : ["input", "output", "json"]
- category.csv        : ["code", "name"]
- furniture.csv       : ["goods_name"]
"""

import os
import json
import math
import random
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model

# ===== 평가지표 관련 =====
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer, util
import evaluate
from sklearn.model_selection import train_test_split


# =========================
# 경로 및 하이퍼파라미터 설정
# =========================

BASE_DIR = "."
TRAIN_CSV_PATH      = os.path.join(BASE_DIR, "data/train_with_json.csv")
CATEGORY_CSV_PATH   = os.path.join(BASE_DIR, "data/Category.csv")
FURNITURE_CSV_PATH  = os.path.join(BASE_DIR, "data/Furniture.csv")

MODEL_NAME   = "K-intelligence/Midm-2.0-Mini-Instruct"
LORA_OUT_DIR = os.path.join(BASE_DIR, "lora_out_midm_full")

MAX_LEN       = 512
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
WARMUP_RATIO  = 0.05
SEED          = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 생성 파라미터 (Repetition Penalty 포함)
GEN_KWARGS = {
    "max_new_tokens": 512,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.18,
}

# =========================
# SFT 학습 전용 SYSTEM PROMPT
# (역할/태스크만 정의, 제약 조건 없음)
# =========================

SYSTEM_PROMPT = (
    "You are an AI assistant specialized in interior design description generation. "
    "Your task is to produce the exact style and structure of outputs shown in the training dataset, "
    "given an input description from the user. "
    "Do not add explanations. Do not alter the intended output format. "
    "Simply learn and follow the patterns demonstrated in the dataset."
)


# =========================
# 공통 유틸 함수
# =========================

def set_seed(seed: int = 42):
    """랜덤 시드 고정"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_category_mapping(path: str) -> Dict[str, str]:
    """
    category.csv 로부터 code → name 매핑 생성
    필수 컬럼: ["code", "name"]
    """
    cat_df = pd.read_csv(path)
    if not {"code", "name"}.issubset(set(cat_df.columns)):
        raise ValueError("category.csv 에 'code', 'name' 컬럼이 필요합니다.")
    return dict(zip(cat_df["code"].astype(str), cat_df["name"].astype(str)))


def replace_codes_in_df(df: pd.DataFrame, mapping: Dict[str, str]) -> pd.DataFrame:
    """
    train_with_json.csv 전체 문자열에 대해
    category 매핑(code → name) 전역 치환
    """
    df_str = df.astype(str)
    return df_str.replace(mapping)


# =========================
# SFT Dataset 정의
# =========================

class SFTDataset(Dataset):
    """
    SFT용 Dataset
    - SYSTEM_PROMPT + [USER] input + [ASSISTANT] output 구조로 하나의 텍스트 생성
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 512):
        self.tokenizer = tokenizer
        self.max_length = max_length

        texts: List[str] = []

        for _, row in df.iterrows():
            user_input    = row["input"]
            target_output = row["output"]

            # Midm Instruct 스타일 프롬프트
            text = (
                f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
                f"[USER]\n{user_input}\n\n"
                f"[ASSISTANT]\n{target_output}</s>"
            )
            texts.append(text)

        enc = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )

        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids      = torch.tensor(self.input_ids[idx], dtype=torch.long)
        attention_mask = torch.tensor(self.attention_mask[idx], dtype=torch.long)
        labels = input_ids.clone()  # 전체 시퀀스를 teacher forcing으로 학습

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# =========================
# Midm + LoRA 로드
# =========================

def load_midm_lora(model_name: str):
    """
    Midm-2.0-Mini-Instruct 4bit + LoRA 로드
    """

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    if tokenizer.eos_token is None:
        tokenizer.eos_token = "</s>"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, tokenizer


# =========================
# 프롬프트/생성 유틸
# =========================

def build_inference_prompt(user_input: str) -> str:
    """
    추론 시 사용할 프롬프트 (학습 시와 동일한 구조 유지)
    """
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
        f"[USER]\n{user_input}\n\n"
        f"[ASSISTANT]\n"
    )


def extract_desc(text: str) -> str:
    """
    모델 출력에서 <DESC> 블록이 있으면 그 안만 사용.
    없으면 전체 텍스트를 DESC로 간주.
    """
    start = text.find("<DESC>")
    end   = text.find("</DESC>")

    if start != -1 and end != -1:
        return text[start + len("<DESC>"): end].strip()
    return text.strip()


def generate_desc_for_batch(
    model,
    tokenizer,
    inputs: List[str],
    gen_kwargs: Dict,
) -> List[str]:
    """
    여러 input에 대해 DESC 텍스트만 생성
    """
    model.eval()
    desc_list: List[str] = []

    with torch.no_grad():
        for user_input in tqdm(inputs, desc="Generating", leave=False):
            prompt = build_inference_prompt(user_input)
            enc = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LEN,
            )

            # ⚠ Midm(LLaMA 계열)은 token_type_ids 사용 안 함 → 제거
            if "token_type_ids" in enc:
                enc.pop("token_type_ids")

            enc = {k: v.to(model.device) for k, v in enc.items()}

            gen_ids = model.generate(
                **enc,
                eos_token_id=tokenizer.eos_token_id,
                **gen_kwargs,
            )

            full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            desc = extract_desc(full_text)
            desc_list.append(desc)

    return desc_list


# =========================
# 평가지표 계산
# =========================

def _norm(s: str) -> str:
    return " ".join(s.strip().split())


def _mean_std(xs: List[float]) -> Tuple[float, float]:
    if not xs:
        return 0.0, 0.0
    m = float(sum(xs) / len(xs))
    v = float(sum((x - m) * (x - m) for x in xs) / len(xs))
    return m, math.sqrt(v)


def compute_metrics_like_script(
    preds: List[str],
    refs: List[str],
    out_detail_path: str = None,
    out_summary_path: str = None,
    tag: str = "",
) -> Dict[str, Dict]:
    """
    사용자가 제공한 평가 스크립트를 그대로 옮긴 형태.
    모든 지표는 0~1 범위에서 계산.
    """

    assert len(preds) == len(refs), "preds / refs 길이가 다릅니다."

    df = pd.DataFrame({
        "predictions": preds,
        "references": refs,
    })

    df = df[~df["predictions"].str.contains(r"\[ERROR\]", na=False)]
    df = df.dropna(subset=["predictions", "references"])

    hyps = df["predictions"].fillna("").astype(str).tolist()
    refs = df["references"].fillna("").astype(str).tolist()

    hyps = [_norm(x) for x in hyps]
    refs = [_norm(x) for x in refs]

    if not hyps:
        print(f"⚠️ ({tag}) 유효한 샘플이 없어 평가를 건너뜁니다.")
        return {}

    print(f"\n✅ ({tag}) 총 {len(hyps)}개 샘플로 5가지 평가지표 계산")

    # --- 1. ROUGE-L ---
    print("\n--- 1. ROUGE-L 계산 중 ---")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rougeL_list = [
        rouge.score(r, h)["rougeL"].fmeasure
        for h, r in tqdm(zip(hyps, refs), desc="ROUGE-L", total=len(hyps))
    ]

    # --- 2. BLEU-4 ---
    print("\n--- 2. BLEU-4 (sacrebleu) 계산 중 ---")
    bleu_corpus = sacrebleu.corpus_bleu(
        hyps, [refs],
        smooth_method="exp",
        smooth_value=None,
        force=True,
        lowercase=False,
        tokenize="none",
    )
    bleu4_corpus = bleu_corpus.score / 100.0  # 0~1

    def sent_bleu(h, r):
        try:
            return sacrebleu.sentence_bleu(
                h, [r],
                smooth_method="exp",
                tokenize="none",
            ).score / 100.0
        except Exception:
            return 0.0

    bleu4_list = [
        sent_bleu(h, r)
        for h, r in tqdm(zip(hyps, refs), desc="Per-sample BLEU-4", total=len(hyps))
    ]

    # --- 3. METEOR ---
    print("\n--- 3. METEOR 계산 중 (HuggingFace evaluate) ---")
    meteor_metric = evaluate.load("meteor")
    meteor_results = meteor_metric.compute(
        predictions=hyps,
        references=[[r] for r in refs],
    )
    meteor_mean = meteor_results["meteor"]  # 0~1

    meteor_list = []
    for h, r in tqdm(zip(hyps, refs), desc="Per-sample METEOR", total=len(hyps)):
        result = meteor_metric.compute(predictions=[h], references=[[r]])
        meteor_list.append(result["meteor"])

    # --- 4. BERTScore ---
    print("\n--- 4. BERTScore 계산 중 (xlm-roberta-large) ---")
    P, R, F1 = bertscore(
        hyps, refs,
        lang="ko",
        model_type="xlm-roberta-large",
        verbose=True,
    )
    berts_f1_list = F1.tolist()
    berts_p_list  = P.tolist()
    berts_r_list  = R.tolist()

    # --- 5. Embedding Cosine ---
    print("\n--- 5. Embedding Cosine Similarity 계산 중 ---")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    emb_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        device=dev,
    )
    emb_h = emb_model.encode(
        hyps, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True
    )
    emb_r = emb_model.encode(
        refs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True
    )
    cosines = util.cos_sim(emb_h, emb_r).diagonal().tolist()

    # detail DataFrame
    out_df = df.copy()
    out_df["rougeL"]   = rougeL_list
    out_df["bleu4"]    = bleu4_list
    out_df["meteor"]   = meteor_list
    out_df["berts_f1"] = berts_f1_list
    out_df["berts_p"]  = berts_p_list
    out_df["berts_r"]  = berts_r_list
    out_df["emb_cos"]  = cosines

    if out_detail_path is not None:
        out_df.to_csv(out_detail_path, index=False, encoding="utf-8-sig")

    # summary json
    summary: Dict[str, Dict] = {}
    for k in ["rougeL", "bleu4", "meteor", "berts_f1", "emb_cos"]:
        m, s = _mean_std(out_df[k].tolist())
        summary[k] = {"mean": round(m, 6), "std": round(s, 6)}

    summary["bleu4_corpus"] = round(bleu4_corpus, 6)

    berts_f1_mean = summary["berts_f1"]["mean"]
    rougeL_mean   = summary["rougeL"]["mean"]
    emb_cos_mean  = summary["emb_cos"]["mean"]

    weighted_score = (
        0.5 * berts_f1_mean +
        0.3 * rougeL_mean +
        0.2 * emb_cos_mean
    )

    summary["WeightedScore"] = round(weighted_score, 6)

    if out_summary_path is not None:
        with open(out_summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 50)
    print(f"  ✨ ({tag}) 평가 결과 저장 완료 ✨")
    print("=" * 50)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    return summary


# =========================
# furniture.csv 기반 가구 추천 (goods_name만 사용)
# =========================

class FurnitureMatcher:
    """
    furniture.csv(goods_name) 기반 가구 매칭
    - goods_name 텍스트를 임베딩으로 변환
    - DESC 임베딩과 cosine similarity로 Top-K 가구명 반환
    """

    def __init__(self, furniture_csv_path: str,
                 embed_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):

        self.df = pd.read_csv(furniture_csv_path)

        if "goods_name" not in self.df.columns:
            raise ValueError("furniture.csv 에 'goods_name' 컬럼이 필요합니다.")

        self.df["goods_name"] = self.df["goods_name"].fillna("").astype(str)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embedder = SentenceTransformer(embed_model_name, device=device)

        self.names = self.df["goods_name"].tolist()
        self.names_emb = self.embedder.encode(
            self.names,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

    def find_similar(self, query: str, top_k: int = 3) -> List[str]:
        """
        DESC 기반으로 가장 유사한 goods_name Top-K 반환
        """
        query_emb = self.embedder.encode(
            [query],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )[0]

        cos_scores = util.cos_sim(query_emb, self.names_emb)[0]
        top_scores, top_indices = torch.topk(cos_scores, k=min(top_k, len(self.df)))

        return [self.names[i] for i in top_indices.cpu().numpy()]


# =========================
# 학습 + 에폭별 평가 + 가구 추천 예시
# =========================

def train_with_metrics_and_furniture():
    set_seed(SEED)

    # 1) 데이터 로드
    df = pd.read_csv(TRAIN_CSV_PATH)

    # 2) category.csv 로 코드값 치환 (code → name)
    mapping = load_category_mapping(CATEGORY_CSV_PATH)
    df = replace_codes_in_df(df, mapping)

    for col in ["input", "output", "json"]:
        if col not in df.columns:
            raise ValueError(f"train_with_json.csv 에 '{col}' 컬럼이 없습니다.")

    # 3) 8:1:1 split
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
    eval_df, val_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED)

    print(f"train: {len(train_df)}, eval: {len(eval_df)}, val: {len(val_df)}")

    # 4) 모델/토크나이저 로드
    model, tokenizer = load_midm_lora(MODEL_NAME)
    model.to(DEVICE)

    # 5) DataLoader 구성
    train_dataset = SFTDataset(train_df, tokenizer, max_length=MAX_LEN)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    # 6) optimizer / scheduler
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
    )

    num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM))
    t_total = num_update_steps_per_epoch * NUM_EPOCHS
    warmup_steps = int(t_total * WARMUP_RATIO)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=t_total,
    )

    os.makedirs(LORA_OUT_DIR, exist_ok=True)

    global_step = 0
    model.train()

    # ===== 학습 루프 (tqdm) =====
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")
        epoch_loss = []

        for step, batch in enumerate(tqdm(train_loader, desc=f"Train Epoch {epoch}")):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            loss = loss / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss.append(loss.item() * GRAD_ACCUM)

        mean_loss = float(np.mean(epoch_loss))
        print(f"[Epoch {epoch}] Train Loss: {mean_loss:.4f}")

        # ----- Epoch 끝에서 eval 세트 평가 -----
        print(f"[Epoch {epoch}] Evaluation 세트 생성 및 지표 계산 시작")
        eval_inputs = eval_df["input"].astype(str).tolist()
        eval_refs   = eval_df["output"].astype(str).tolist()

        preds = generate_desc_for_batch(
            model,
            tokenizer,
            eval_inputs,
            GEN_KWARGS,
        )

        detail_path  = os.path.join(LORA_OUT_DIR, f"metrics_epoch{epoch}_detail.csv")
        summary_path = os.path.join(LORA_OUT_DIR, f"metrics_epoch{epoch}_summary.json")

        compute_metrics_like_script(
            preds,
            eval_refs,
            out_detail_path=detail_path,
            out_summary_path=summary_path,
            tag=f"Epoch {epoch} / Eval",
        )

        # 에폭별 LoRA 저장
        epoch_dir = os.path.join(LORA_OUT_DIR, f"epoch{epoch}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)
        tokenizer.save_pretrained(epoch_dir)

    # ===== 학습 종료 후 validation 세트 평가 =====
    print("\n===== Training Finished. Validation 세트 평가 시작 =====")
    val_inputs = val_df["input"].astype(str).tolist()
    val_refs   = val_df["output"].astype(str).tolist()

    preds_val = generate_desc_for_batch(
        model,
        tokenizer,
        val_inputs,
        GEN_KWARGS,
    )

    val_detail  = os.path.join(LORA_OUT_DIR, "metrics_val_detail.csv")
    val_summary = os.path.join(LORA_OUT_DIR, "metrics_val_summary.json")

    compute_metrics_like_script(
        preds_val,
        val_refs,
        out_detail_path=val_detail,
        out_summary_path=val_summary,
        tag="Validation",
    )

    # 최종 LoRA 저장
    model.save_pretrained(LORA_OUT_DIR)
    tokenizer.save_pretrained(LORA_OUT_DIR)

    print("\n모든 학습 및 평가 완료.")

    # ===== 예시 인퍼런스 + 가구 추천 =====
    try:
        print("\n===== 예시 인퍼런스 + furniture.csv 기반 가구 추천 =====")
        matcher = FurnitureMatcher(FURNITURE_CSV_PATH)

        example_input = "화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간"
        print(f"[USER 입력 예시]\n{example_input}\n")

        example_preds = generate_desc_for_batch(
            model,
            tokenizer,
            [example_input],
            GEN_KWARGS,
        )
        example_desc = example_preds[0]
        print("[생성된 DESC]\n", example_desc, "\n")

        similar_furns = matcher.find_similar(example_desc, top_k=3)
        print("[유사 가구 Top-3]")
        for name in similar_furns:
            print("-", name)

    except Exception as e:
        print("\n가구 추천 예시에서 오류 발생:", e)


if __name__ == "__main__":
    train_with_metrics_and_furniture()

train: 856, eval: 107, val: 108


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

[USER 입력 예시]
화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간



Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[생성된 DESC]
 <s>[SYSTEM]
You are an AI assistant specialized in interior design description generation. Your task is to produce the exact style and structure of outputs shown in the training dataset, given an input description from the user. Do not add explanations. Do not alter the intended output format. Simply learn and follow the patterns demonstrated in the dataset.

[USER]
화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간

[ASSISTANT]
거실을 따뜻하고 아늑하게 만들어 주는 화이트&우드 인테리어입니다. 벽면에는 그레이 컬러로 포인트를 주어 단조로움을 피했고, 천장은 직접 조명보다는 간접 조명으로 은은하고 고급스러운 분위기를 연출했습니다.</s> 

[유사 가구 Top-3]
- 티오 각도조절책상 세트 (옵션 택1)
- 시그니처 스케치 붙박이장 맞춤설계
- 시그니처 스케치 라인 붙박이장 맞춤설계


---
# ver2

In [4]:
!pip install -r requirements.txt

  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached peft-0.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached sacrebleu-2.5.1-py3-none-any.whl.metadata (51 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached accelerate-1.11.0-py3-none-any.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.met

In [ ]:
"""
Midm-2.0-Mini-Instruct + LoRA SFT 학습 스크립트 (개선 버전)

개선 사항
1) category.csv(code, name) 치환 후 학습
2) train_with_json.csv 의 output + json 을 결합한 전체 타깃을 학습/평가에 사용
3) 학습용 타깃과 평가용 reference 를 동일한 build_target_text(row) 로 생성
4) Epoch별 eval / 최종 val 에서 BLEU / ROUGE-L / METEOR / BERTScore / Embedding Cosine / WeightedScore 계산
5) FurnitureMatcher 는 여전히 DESC 기반으로 goods_name Top-K 추천
"""

import os
import json
import math
import random
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model

# ===== 평가지표 관련 =====
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer, util
import evaluate
from sklearn.model_selection import train_test_split


# =========================
# 경로 및 하이퍼파라미터 설정
# =========================

BASE_DIR = "."
TRAIN_CSV_PATH      = os.path.join(BASE_DIR, "data/train_with_json.csv")
CATEGORY_CSV_PATH   = os.path.join(BASE_DIR, "data/Category.csv")
FURNITURE_CSV_PATH  = os.path.join(BASE_DIR, "data/Furniture.csv")

MODEL_NAME   = "K-intelligence/Midm-2.0-Mini-Instruct"
LORA_OUT_DIR = os.path.join(BASE_DIR, "lora_out_midm_full")

MAX_LEN       = 512
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
WARMUP_RATIO  = 0.05
SEED          = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 생성 파라미터 (Repetition Penalty 포함)
GEN_KWARGS = {
    "max_new_tokens": 512,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.18,
}

# =========================
# SFT 학습 전용 SYSTEM PROMPT
# =========================

SYSTEM_PROMPT = (
    "You are an AI assistant specialized in interior design description and image prompt generation. "
    "Given an input condition from the user, you must reproduce the exact style and structure of the target outputs "
    "shown in the training dataset, including both the natural language description and the JSON block for image models. "
    "Do not add explanations. Do not change the output format. "
    "Always follow the patterns demonstrated in the dataset."
)


# =========================
# 공통 유틸 함수
# =========================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_category_mapping(path: str) -> Dict[str, str]:
    cat_df = pd.read_csv(path)
    if not {"code", "name"}.issubset(set(cat_df.columns)):
        raise ValueError("category.csv 에 'code', 'name' 컬럼이 필요합니다.")
    return dict(zip(cat_df["code"].astype(str), cat_df["name"].astype(str)))


def replace_codes_in_df(df: pd.DataFrame, mapping: Dict[str, str]) -> pd.DataFrame:
    df_str = df.astype(str)
    return df_str.replace(mapping)


def build_target_text(row: pd.Series) -> str:
    """
    한 샘 태스크에서 모델이 최종적으로 생성해야 할 타깃 텍스트 정의.
    - output: 인테리어 설명 (또는 <DESC> 블록 포함)
    - json  : image model prompt 로 들어갈 JSON 문자열
    둘을 결합하여 하나의 타깃 텍스트로 사용.
    """
    out = str(row.get("output", "")).strip()
    js  = str(row.get("json", "")).strip()

    # DESC 블록이 이미 있다면 그대로 쓰고, 없으면 래핑해도 됨
    if "<DESC>" in out and "</DESC>" in out:
        desc_part = out
    else:
        # 너무 짧으면 학습에 도움 안 되므로 필터링 가능
        desc_part = f"<DESC>{out}</DESC>" if out else ""

    # JSON 블록 래핑
    if js:
        if "<JSON>" in js and "</JSON>" in js:
            json_part = js
        else:
            json_part = f"<JSON>{js}</JSON>"
    else:
        json_part = ""

    target = (desc_part + "\n" + json_part).strip()
    return target


# =========================
# SFT Dataset 정의
# =========================

class SFTDataset(Dataset):
    """
    SFT용 Dataset
    - SYSTEM_PROMPT + [USER] input + [ASSISTANT] target_text 구조로 학습
    - target_text = build_target_text(row)  (output + json 결합)
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 512):
        self.tokenizer = tokenizer
        self.max_length = max_length

        texts: List[str] = []

        for _, row in df.iterrows():
            user_input   = str(row["input"])
            target_text  = build_target_text(row)

            # 너무 짧거나 빈 타깃은 스킵(데이터 노이즈 방지)
            if len(target_text) < 10:
                continue

            text = (
                f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
                f"[USER]\n{user_input}\n\n"
                f"[ASSISTANT]\n{target_text}</s>"
            )
            texts.append(text)

        if not texts:
            raise ValueError("유효한 학습 샘플이 없습니다. 데이터/전처리를 확인하세요.")

        enc = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )

        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids      = torch.tensor(self.input_ids[idx], dtype=torch.long)
        attention_mask = torch.tensor(self.attention_mask[idx], dtype=torch.long)
        labels = input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# =========================
# Midm + LoRA 로드
# =========================

def load_midm_lora(model_name: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    if tokenizer.eos_token is None:
        tokenizer.eos_token = "</s>"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, tokenizer


# =========================
# 프롬프트/생성 유틸
# =========================

def build_inference_prompt(user_input: str) -> str:
    """
    추론 시 사용할 프롬프트 (학습 시와 동일한 구조)
    """
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
        f"[USER]\n{user_input}\n\n"
        f"[ASSISTANT]\n"
    )


def extract_desc(text: str) -> str:
    """
    생성 결과에서 DESC 블록만 추출 (가구 추천용)
    """
    start = text.find("<DESC>")
    end   = text.find("</DESC>")

    if start != -1 and end != -1:
        return text[start + len("<DESC>"): end].strip()
    return text.strip()


def generate_full_for_batch(
    model,
    tokenizer,
    inputs: List[str],
    gen_kwargs: Dict,
) -> List[str]:
    """
    여러 input에 대해 전체 출력(설명 + JSON)을 생성
    """
    model.eval()
    outs: List[str] = []

    with torch.no_grad():
        for user_input in tqdm(inputs, desc="Generating", leave=False):
            prompt = build_inference_prompt(user_input)
            enc = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LEN,
            )

            # Midm(LLaMA 계열)은 token_type_ids 사용 X
            if "token_type_ids" in enc:
                enc.pop("token_type_ids")

            enc = {k: v.to(model.device) for k, v in enc.items()}

            gen_ids = model.generate(
                **enc,
                eos_token_id=tokenizer.eos_token_id,
                **gen_kwargs,
            )

            full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            # [ASSISTANT] 이후만 사용 (혹시 프롬프트까지 섞여 나올 경우)
            cut = full_text.split("[ASSISTANT]", 1)
            if len(cut) == 2:
                out_txt = cut[1].strip()
            else:
                out_txt = full_text.strip()

            outs.append(out_txt)

    return outs


# =========================
# 평가지표 계산
# =========================

def _norm(s: str) -> str:
    return " ".join(s.strip().split())


def _mean_std(xs: List[float]) -> Tuple[float, float]:
    if not xs:
        return 0.0, 0.0
    m = float(sum(xs) / len(xs))
    v = float(sum((x - m) * (x - m) for x in xs) / len(xs))
    return m, math.sqrt(v)


def compute_metrics_like_script(
    preds: List[str],
    refs: List[str],
    out_detail_path: str = None,
    out_summary_path: str = None,
    tag: str = "",
) -> Dict[str, Dict]:
    """
    모든 지표를 0~1 스케일로 계산 (사용자 스크립트 동일 로직)
    """

    assert len(preds) == len(refs), "preds / refs 길이가 다릅니다."

    df = pd.DataFrame({
        "predictions": preds,
        "references": refs,
    })

    df = df[~df["predictions"].str.contains(r"\[ERROR\]", na=False)]
    df = df.dropna(subset=["predictions", "references"])

    hyps = df["predictions"].fillna("").astype(str).tolist()
    refs = df["references"].fillna("").astype(str).tolist()

    hyps = [_norm(x) for x in hyps]
    refs = [_norm(x) for x in refs]

    if not hyps:
        print(f"⚠️ ({tag}) 유효한 샘플이 없어 평가를 건너뜁니다.")
        return {}

    print(f"\n✅ ({tag}) 총 {len(hyps)}개 샘플로 5가지 평가지표 계산")

    # --- 1. ROUGE-L ---
    print("\n--- 1. ROUGE-L 계산 중 ---")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rougeL_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc="ROUGE-L", total=len(hyps)):
        rougeL_list.append(rouge.score(r, h)["rougeL"].fmeasure)

    # --- 2. BLEU-4 ---
    print("\n--- 2. BLEU-4 (sacrebleu) 계산 중 ---")
    bleu_corpus = sacrebleu.corpus_bleu(
        hyps, [refs],
        smooth_method="exp",
        smooth_value=None,
        force=True,
        lowercase=False,
        tokenize="none",
    )
    bleu4_corpus = bleu_corpus.score / 100.0  # 0~1

    def sent_bleu(h, r):
        try:
            return sacrebleu.sentence_bleu(
                h, [r],
                smooth_method="exp",
                tokenize="none",
            ).score / 100.0
        except Exception:
            return 0.0

    bleu4_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc="Per-sample BLEU-4", total=len(hyps)):
        bleu4_list.append(sent_bleu(h, r))

    # --- 3. METEOR ---
    print("\n--- 3. METEOR 계산 중 (HuggingFace evaluate) ---")
    meteor_metric = evaluate.load("meteor")
    meteor_results = meteor_metric.compute(
        predictions=hyps,
        references=[[r] for r in refs],
    )
    meteor_mean = meteor_results["meteor"]

    meteor_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc="Per-sample METEOR", total=len(hyps)):
        result = meteor_metric.compute(predictions=[h], references=[[r]])
        meteor_list.append(result["meteor"])

    # --- 4. BERTScore ---
    print("\n--- 4. BERTScore 계산 중 (xlm-roberta-large) ---")
    P, R, F1 = bertscore(
        hyps, refs,
        lang="ko",
        model_type="xlm-roberta-large",
        verbose=True,
    )
    berts_f1_list = F1.tolist()
    berts_p_list  = P.tolist()
    berts_r_list  = R.tolist()

    # --- 5. Embedding Cosine ---
    print("\n--- 5. Embedding Cosine Similarity 계산 중 ---")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    emb_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        device=dev,
    )
    emb_h = emb_model.encode(
        hyps, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True
    )
    emb_r = emb_model.encode(
        refs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True
    )
    cosines = util.cos_sim(emb_h, emb_r).diagonal().tolist()

    out_df = df.copy()
    out_df["rougeL"]   = rougeL_list
    out_df["bleu4"]    = bleu4_list
    out_df["meteor"]   = meteor_list
    out_df["berts_f1"] = berts_f1_list
    out_df["berts_p"]  = berts_p_list
    out_df["berts_r"]  = berts_r_list
    out_df["emb_cos"]  = cosines

    if out_detail_path is not None:
        out_df.to_csv(out_detail_path, index=False, encoding="utf-8-sig")

    summary: Dict[str, Dict] = {}
    for k in ["rougeL", "bleu4", "meteor", "berts_f1", "emb_cos"]:
        m, s = _mean_std(out_df[k].tolist())
        summary[k] = {"mean": round(m, 6), "std": round(s, 6)}

    summary["bleu4_corpus"] = round(bleu4_corpus, 6)

    berts_f1_mean = summary["berts_f1"]["mean"]
    rougeL_mean   = summary["rougeL"]["mean"]
    emb_cos_mean  = summary["emb_cos"]["mean"]

    weighted_score = (
        0.5 * berts_f1_mean +
        0.3 * rougeL_mean +
        0.2 * emb_cos_mean
    )

    summary["WeightedScore"] = round(weighted_score, 6)

    if out_summary_path is not None:
        with open(out_summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 50)
    print(f"  ✨ ({tag}) 평가 결과 저장 완료 ✨")
    print("=" * 50)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    return summary


# =========================
# furniture.csv 기반 가구 추천
# =========================

class FurnitureMatcher:
    """
    furniture.csv(goods_name) 기반 가구 매칭
    """

    def __init__(self, furniture_csv_path: str,
                 embed_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):

        self.df = pd.read_csv(furniture_csv_path)

        if "goods_name" not in self.df.columns:
            raise ValueError("furniture.csv 에 'goods_name' 컬럼이 필요합니다.")

        self.df["goods_name"] = self.df["goods_name"].fillna("").astype(str)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embedder = SentenceTransformer(embed_model_name, device=device)

        self.names = self.df["goods_name"].tolist()
        self.names_emb = self.embedder.encode(
            self.names,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

    def find_similar(self, query: str, top_k: int = 3) -> List[str]:
        query_emb = self.embedder.encode(
            [query],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )[0]

        cos_scores = util.cos_sim(query_emb, self.names_emb)[0]
        top_scores, top_indices = torch.topk(cos_scores, k=min(top_k, len(self.df)))

        return [self.names[i] for i in top_indices.cpu().numpy()]


# =========================
# 학습 + 에폭별 평가 + 가구 추천 예시
# =========================

def train_with_metrics_and_furniture():
    set_seed(SEED)

    # 1) 데이터 로드
    df = pd.read_csv(TRAIN_CSV_PATH)

    # 2) category.csv 로 코드값 치환
    mapping = load_category_mapping(CATEGORY_CSV_PATH)
    df = replace_codes_in_df(df, mapping)

    for col in ["input", "output", "json"]:
        if col not in df.columns:
            raise ValueError(f"train_with_json.csv 에 '{col}' 컬럼이 없습니다.")

    # 3) 8:1:1 split
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
    eval_df, val_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED)

    print(f"train: {len(train_df)}, eval: {len(eval_df)}, val: {len(val_df)}")

    # 4) 모델/토크나이저 로드
    model, tokenizer = load_midm_lora(MODEL_NAME)
    model.to(DEVICE)

    # 5) DataLoader
    train_dataset = SFTDataset(train_df, tokenizer, max_length=MAX_LEN)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    # 6) optimizer / scheduler
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
    )

    num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM))
    t_total = num_update_steps_per_epoch * NUM_EPOCHS
    warmup_steps = int(total * WARMUP_RATIO) if (total := t_total) > 0 else 0

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=t_total,
    )

    os.makedirs(LORA_OUT_DIR, exist_ok=True)

    global_step = 0
    model.train()

    # ===== 학습 루프 =====
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")
        epoch_loss = []

        for step, batch in enumerate(tqdm(train_loader, desc=f"Train Epoch {epoch}")):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            loss = loss / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss.append(loss.item() * GRAD_ACCUM)

        mean_loss = float(np.mean(epoch_loss))
        print(f"[Epoch {epoch}] Train Loss: {mean_loss:.4f}")

        # ----- Epoch별 eval -----
        print(f"[Epoch {epoch}] Evaluation 세트 생성 및 지표 계산")
        eval_inputs = eval_df["input"].astype(str).tolist()
        eval_refs   = [build_target_text(row) for _, row in eval_df.iterrows()]

        preds = generate_full_for_batch(
            model,
            tokenizer,
            eval_inputs,
            GEN_KWARGS,
        )

        detail_path  = os.path.join(LORA_OUT_DIR, f"metrics_epoch{epoch}_detail.csv")
        summary_path = os.path.join(LORA_OUT_DIR, f"metrics_epoch{epoch}_summary.json")

        compute_metrics_like_script(
            preds,
            eval_refs,
            out_detail_path=detail_path,
            out_summary_path=summary_path,
            tag=f"Epoch {epoch} / Eval",
        )

        epoch_dir = os.path.join(LORA_OUT_DIR, f"epoch{epoch}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)
        tokenizer.save_pretrained(epoch_dir)

    # ===== 최종 Validation 평가 =====
    print("\n===== Training Finished. Validation 세트 평가 시작 =====")
    val_inputs = val_df["input"].astype(str).tolist()
    val_refs   = [build_target_text(row) for _, row in val_df.iterrows()]

    preds_val = generate_full_for_batch(
        model,
        tokenizer,
        val_inputs,
        GEN_KWARGS,
    )

    val_detail  = os.path.join(LORA_OUT_DIR, "metrics_val_detail.csv")
    val_summary = os.path.join(LORA_OUT_DIR, "metrics_val_summary.json")

    compute_metrics_like_script(
        preds_val,
        val_refs,
        out_detail_path=val_detail,
        out_summary_path=val_summary,
        tag="Validation",
    )

    model.save_pretrained(LORA_OUT_DIR)
    tokenizer.save_pretrained(LORA_OUT_DIR)

    print("\n모든 학습 및 평가 완료.")

    # ===== 예시 인퍼런스 + 가구 추천 =====
    try:
        print("\n===== 예시 인퍼런스 + furniture.csv 기반 가구 추천 =====")
        matcher = FurnitureMatcher(FURNITURE_CSV_PATH)

        example_input = "화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간"
        print(f"[USER 입력 예시]\n{example_input}\n")

        example_full = generate_full_for_batch(
            model,
            tokenizer,
            [example_input],
            GEN_KWARGS,
        )[0]
        print("[생성된 전체 출력]\n", example_full, "\n")

        desc_only = extract_desc(example_full)
        similar_furns = matcher.find_similar(desc_only, top_k=3)
        print("[유사 가구 Top-3]")
        for name in similar_furns:
            print("-", name)

    except Exception as e:
        print("\n가구 추천 예시에서 오류 발생:", e)


if __name__ == "__main__":
    train_with_metrics_and_furniture()

train: 856, eval: 107, val: 108
trainable params: 6,684,672 || all params: 2,312,201,984 || trainable%: 0.2891

===== Epoch 1/3 =====


Train Epoch 1:   0%|          | 0/856 [00:00<?, ?it/s]

[Epoch 1] Train Loss: 2.1126
[Epoch 1] Evaluation 세트 생성 및 지표 계산


Generating:   0%|          | 0/107 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

...

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

[USER 입력 예시]
화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간



Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[생성된 전체 출력]
 <DESC>주방은 밝은 화이트 톤에 그레이 컬러를 포인트로 사용하여 고급스러운 느낌을 더했습니다! 벽면 수납장으로 깔끔한 분위기 연출이 가능하며, 냉장고장 아래에는 선반형 수납공간까지 넣어 실용성도 높였습니다.</DESC>
<JSON>{"keywords": ["주방", "고급스러운", "깔끔한"], "style": ["모던", "미니멀리즘"], "colors": ["밝은 화이트", "그레이"], "lighting": ["자연광", "부드러운 조명"], "space": ["수납공간", "조리대"], "furniture": ["냉장고", "선반", "서랍식 서랍"], "mood": ["편안함", "실용성"], "composition": "벽면 수납장과 조리대가 조화롭게 배치된 주방 구조", "camera_angle": "정면", "camera_params": {"fov": "50mm", "aperture": "2.8", "iso": "100", "shutter": "1/60"}, "negative": []}</SJ>


</user> 

[유사 가구 Top-3]
- 모디 주방수납 하부장 100cm 가전형
- 모디 주방수납 카페장 100cm 가전형 (6종 택1)
- 라임 주방수납 가전수납장 80cm (와인잔걸이 포함) (2종 택1)


---
# MERGE

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "K-intelligence/Midm-2.0-Mini-Instruct"
LORA_PATH       = "./lora_out_midm_full/epoch3"      # ← 네가 선택한 epoch 폴더
MERGED_OUT      = "./merged_midm_epoch3"             # ← 병합된 최종 모델 저장 경로

def main():
    os.makedirs(MERGED_OUT, exist_ok=True)

    print("1) 베이스 모델 로드 중...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16,      # GPU 있으면 float16 권장
        device_map="auto",
        trust_remote_code=True,
    )

    print("2) LoRA 어댑터 로드 및 결합 중...")
    lora_model = PeftModel.from_pretrained(
        base_model,
        LORA_PATH,
    )

    print("3) LoRA 가중치 병합(merge_and_unload) 중...")
    merged_model = lora_model.merge_and_unload()   # base + LoRA → 하나의 모델로 통합

    print("4) 병합된 모델 저장 중...")
    merged_model.save_pretrained(MERGED_OUT)

    # 토크나이저도 함께 저장
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=False)
    tokenizer.save_pretrained(MERGED_OUT)

    print("\n✅ LoRA 병합 완료")
    print(f"   - 병합 모델 경로: {MERGED_OUT}")
    print("   AutoModelForCausalLM.from_pretrained(MERGED_OUT) 사용 가능.")

if __name__ == "__main__":
    main()

1) 베이스 모델 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


2) LoRA 어댑터 로드 및 결합 중...
3) LoRA 가중치 병합(merge_and_unload) 중...
4) 병합된 모델 저장 중...

✅ LoRA 병합 완료
   - 병합 모델 경로: ./merged_midm_epoch3
   AutoModelForCausalLM.from_pretrained(MERGED_OUT) 사용 가능.


---
# EVAL - v2
- 성능 분할 평가

In [11]:
"""
DESC / JSON 분리 평가용 개선 버전

- 입력: eval_results*.csv (extracted_response, expected_output)
- 출력:
  - metrics_detail_split.csv  : full/desc/json 기준 per-sample 지표
  - metrics_summary_split.json: full/desc/json 기준 요약 지표
"""

import os
import json
import math
import re
from typing import List, Dict, Tuple

import torch
import pandas as pd
from tqdm.auto import tqdm
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer, util
import evaluate  # METEOR

# ======================
# 경로 설정
# ======================

EVAL_PATH   = "./lora_out_midm_full/metrics_val_detail.csv"              # 네가 쓰는 eval csv 경로로 변경
OUT_DETAIL  = "./metrics_detail_split.csv"
OUT_SUMMARY = "./metrics_summary_split.json"


# ======================
# 유틸 함수
# ======================

def _norm(s: str) -> str:
    return " ".join(str(s).strip().split())


def _mean_std(xs: List[float]) -> Tuple[float, float]:
    if not xs:
        return 0.0, 0.0
    m = float(sum(xs) / len(xs))
    v = float(sum((x - m) * (x - m) for x in xs) / len(xs))
    return m, math.sqrt(v)


def extract_block(text: str, tag: str) -> str:
    """
    <TAG> ... </TAG> 사이 문자열 추출
    예: tag="DESC", "JSON"
    """
    if not isinstance(text, str):
        return ""
    pattern = rf"<{tag}>(.*?)</{tag}>"
    m = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return ""


def safe_load_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None


# ======================
# 텍스트 기준 공통 지표 계산
# (ROUGE-L / BLEU-4 / METEOR / BERTScore / Embedding Cosine)
# ======================

def compute_text_metrics(
    hyps: List[str],
    refs: List[str],
    tag: str,
    emb_model=None,
) -> Tuple[pd.DataFrame, Dict[str, Dict]]:
    """
    hyps, refs: 이미 _norm 처리된 텍스트 리스트
    return: (per-sample df, summary dict)
    """
    assert len(hyps) == len(refs)
    n = len(hyps)
    print(f"\n✅ ({tag}) {n}개 샘플에 대해 텍스트 기준 지표 계산")

    # 1. ROUGE-L
    print(f"\n[{tag}] 1. ROUGE-L 계산 중 ...")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rougeL_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc=f"{tag} - ROUGE-L", total=n):
        rougeL_list.append(rouge.score(r, h)["rougeL"].fmeasure)

    # 2. BLEU-4
    print(f"\n[{tag}] 2. BLEU-4 (sacrebleu) 계산 중 ...")
    bleu_corpus = sacrebleu.corpus_bleu(
        hyps, [refs],
        smooth_method="exp",
        smooth_value=None,
        force=True,
        lowercase=False,
        tokenize="none",
    )
    bleu4_corpus = bleu_corpus.score / 100.0

    def sent_bleu(h, r):
        try:
            return sacrebleu.sentence_bleu(
                h, [r],
                smooth_method="exp",
                tokenize="none",
            ).score / 100.0
        except Exception:
            return 0.0

    bleu4_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc=f"{tag} - Per-sample BLEU-4", total=n):
        bleu4_list.append(sent_bleu(h, r))

    # 3. METEOR
    print(f"\n[{tag}] 3. METEOR 계산 중 (evaluate: meteor) ...")
    meteor_metric = evaluate.load("meteor")
    meteor_results = meteor_metric.compute(
        predictions=hyps,
        references=[[r] for r in refs],
    )
    meteor_mean = meteor_results["meteor"]  # 전체 평균

    meteor_list = []
    for h, r in tqdm(list(zip(hyps, refs)), desc=f"{tag} - Per-sample METEOR", total=n):
        result = meteor_metric.compute(predictions=[h], references=[[r]])
        meteor_list.append(result["meteor"])

    # 4. BERTScore
    print(f"\n[{tag}] 4. BERTScore 계산 중 (xlm-roberta-large) ...")
    P, R, F1 = bertscore(
        hyps, refs,
        lang="ko",
        model_type="xlm-roberta-large",
        verbose=True,
    )
    berts_f1_list = F1.tolist()
    berts_p_list  = P.tolist()
    berts_r_list  = R.tolist()

    # 5. Embedding Cosine
    print(f"\n[{tag}] 5. Embedding Cosine (SentenceTransformer) 계산 중 ...")
    if emb_model is None:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        emb_model = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=dev,
        )

    emb_h = emb_model.encode(
        hyps, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=True
    )
    emb_r = emb_model.encode(
        refs, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=True
    )
    cosines = util.cos_sim(emb_h, emb_r).diagonal().tolist()

    # per-sample df
    df = pd.DataFrame({
        "predictions": hyps,
        "references":  refs,
        "rougeL":      rougeL_list,
        "bleu4":       bleu4_list,
        "meteor":      meteor_list,
        "berts_f1":    berts_f1_list,
        "berts_p":     berts_p_list,
        "berts_r":     berts_r_list,
        "emb_cos":     cosines,
    })

    # summary
    summary: Dict[str, Dict] = {}
    for k in ["rougeL", "bleu4", "meteor", "berts_f1", "emb_cos"]:
        m, s = _mean_std(df[k].tolist())
        summary[k] = {"mean": round(m, 6), "std": round(s, 6)}

    summary["bleu4_corpus"] = round(bleu4_corpus, 6)

    # WeightedScore (텍스트 기준)
    berts_f1_mean = summary["berts_f1"]["mean"]
    rougeL_mean   = summary["rougeL"]["mean"]
    emb_cos_mean  = summary["emb_cos"]["mean"]
    weighted_score = (
        0.5 * berts_f1_mean +
        0.3 * rougeL_mean +
        0.2 * emb_cos_mean
    )
    summary["WeightedScore"] = round(weighted_score, 6)

    print("\n" + "=" * 50)
    print(f"  ✨ ({tag}) 텍스트 지표 계산 완료 ✨")
    print("=" * 50)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    return df, summary


# ======================
# JSON 전용 지표 계산
#  - json_exact   : JSON 블록 문자열 완전 일치 여부(0/1)
#  - json_key_f1  : JSON key 집합 기준 F1
#  - json_rougeL  : JSON 문자열 ROUGE-L
#  - json_val_cos : JSON 문자열 임베딩 코사인
# ======================

def compute_json_metrics(
    json_hyps: List[str],
    json_refs: List[str],
    tag: str,
    emb_model=None,
) -> Tuple[pd.DataFrame, Dict[str, Dict]]:
    assert len(json_hyps) == len(json_refs)
    n = len(json_hyps)
    print(f"\n✅ ({tag}) {n}개 샘플에 대해 JSON 지표 계산")

    # 문자열 정규화
    hyps_norm = [_norm(x) for x in json_hyps]
    refs_norm = [_norm(x) for x in json_refs]

    # 1) exact match (문자열 동일 여부)
    json_exact = [1.0 if h == r and h != "" else 0.0 for h, r in zip(hyps_norm, refs_norm)]

    # 2) key-level F1
    json_key_f1 = []
    for h, r in tqdm(list(zip(json_hyps, json_refs)), desc=f"{tag} - JSON key F1", total=n):
        hp = safe_load_json(h)
        rf = safe_load_json(r)
        if not isinstance(hp, dict) or not isinstance(rf, dict):
            json_key_f1.append(0.0)
            continue
        k_h = set(hp.keys())
        k_r = set(rf.keys())
        if not k_h or not k_r:
            json_key_f1.append(0.0)
            continue
        inter = len(k_h & k_r)
        prec = inter / len(k_h) if len(k_h) > 0 else 0.0
        rec  = inter / len(k_r) if len(k_r) > 0 else 0.0
        if prec + rec == 0:
            f1 = 0.0
        else:
            f1 = 2 * prec * rec / (prec + rec)
        json_key_f1.append(float(f1))

    # 3) JSON 문자열 ROUGE-L
    print(f"\n[{tag}] JSON ROUGE-L 계산 중 ...")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    json_rougeL = []
    for h, r in tqdm(list(zip(hyps_norm, refs_norm)), desc=f"{tag} - JSON ROUGE-L", total=n):
        if h == "" or r == "":
            json_rougeL.append(0.0)
            continue
        json_rougeL.append(rouge.score(r, h)["rougeL"].fmeasure)

    # 4) JSON 문자열 Embedding Cosine
    print(f"\n[{tag}] JSON Embedding Cosine 계산 중 ...")
    if emb_model is None:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        emb_model = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=dev,
        )
    emb_h = emb_model.encode(
        hyps_norm, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=True
    )
    emb_r = emb_model.encode(
        refs_norm, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=True
    )
    json_val_cos = util.cos_sim(emb_h, emb_r).diagonal().tolist()

    df = pd.DataFrame({
        "json_pred":    json_hyps,
        "json_ref":     json_refs,
        "json_exact":   json_exact,
        "json_key_f1":  json_key_f1,
        "json_rougeL":  json_rougeL,
        "json_val_cos": json_val_cos,
    })

    summary: Dict[str, Dict] = {}
    for k in ["json_exact", "json_key_f1", "json_rougeL", "json_val_cos"]:
        m, s = _mean_std(df[k].tolist())
        summary[k] = {"mean": round(m, 6), "std": round(s, 6)}

    # JSON 전용 가중합 (참고용)
    json_val_cos_mean = summary["json_val_cos"]["mean"]
    json_rougeL_mean  = summary["json_rougeL"]["mean"]
    json_exact_mean   = summary["json_exact"]["mean"]
    weighted_json = (
        0.4 * json_val_cos_mean +
        0.4 * json_rougeL_mean +
        0.2 * json_exact_mean
    )
    summary["WeightedScore_json"] = round(weighted_json, 6)

    print("\n" + "=" * 50)
    print(f"  ✨ ({tag}) JSON 지표 계산 완료 ✨")
    print("=" * 50)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    return df, summary


# ======================
# 메인 루틴
# ======================

def main():
    # 1) CSV 로드
    try:
        df_raw = pd.read_csv(EVAL_PATH, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_raw = pd.read_csv(EVAL_PATH, encoding="utf-8")

    # --- 컬럼 자동 인식 (둘 중 하나 형태면 OK) ---
    cols = set(df_raw.columns)

    # case 1: extracted_response / expected_output
    if {"extracted_response", "expected_output"}.issubset(cols):
        df = df_raw.rename(columns={
            "extracted_response": "predictions",
            "expected_output":    "references",
        })

    # case 2: predictions / references
    elif {"predictions", "references"}.issubset(cols):
        df = df_raw.copy()   # 이미 올바른 이름이므로 그대로 사용

    else:
        raise ValueError(
            "CSV에 필요한 컬럼이 없습니다. "
            "다음 중 하나를 포함해야 합니다:\n"
            "1) extracted_response, expected_output\n"
            "2) predictions, references"
        )

    # 에러/결측 제거
    df = df[~df["predictions"].astype(str).str.contains(r"\[ERROR\]", na=False)]
    df = df.dropna(subset=["predictions", "references"])

    if df.empty:
        print("⚠️ 유효한 샘플이 없어 평가를 건너뜁니다.")
        return

    # full 텍스트
    full_hyps = df["predictions"].astype(str).tolist()
    full_refs = df["references"].astype(str).tolist()

    # DESC, JSON 분리
    desc_hyps = [extract_block(t, "DESC") for t in full_hyps]
    desc_refs = [extract_block(t, "DESC") for t in full_refs]
    json_hyps = [extract_block(t, "JSON") for t in full_hyps]
    json_refs = [extract_block(t, "JSON") for t in full_refs]

    # 정규화
    full_hyps_norm = [_norm(x) for x in full_hyps]
    full_refs_norm = [_norm(x) for x in full_refs]
    desc_hyps_norm = [_norm(x) for x in desc_hyps]
    desc_refs_norm = [_norm(x) for x in desc_refs]

    # 공용 embedding 모델 (재사용)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    shared_emb_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        device=dev,
    )

    # 2) full 텍스트 기준 지표
    full_df, full_summary = compute_text_metrics(
        full_hyps_norm, full_refs_norm,
        tag="FULL",
        emb_model=shared_emb_model,
    )

    # 3) DESC 기준 지표
    #    (DESC가 비어있는 경우는 필터링)
    desc_mask = [len(x) > 0 and len(y) > 0 for x, y in zip(desc_hyps_norm, desc_refs_norm)]
    desc_h = [h for h, m in zip(desc_hyps_norm, desc_mask) if m]
    desc_r = [r for r, m in zip(desc_refs_norm, desc_mask) if m]

    if desc_h:
        desc_df, desc_summary = compute_text_metrics(
            desc_h, desc_r,
            tag="DESC",
            emb_model=shared_emb_model,
        )
    else:
        desc_df = pd.DataFrame()
        desc_summary = {}

    # 4) JSON 기준 지표
    json_df, json_summary = compute_json_metrics(
        json_hyps, json_refs,
        tag="JSON",
        emb_model=shared_emb_model,
    )

    # 5) detail / summary 저장
    # detail: full 기준 텍스트 + JSON 지표를 하나로 합쳐서 저장
    detail_df = df.copy()
    # full text metrics
    for col in ["rougeL", "bleu4", "meteor", "berts_f1", "berts_p", "berts_r", "emb_cos"]:
        detail_df[f"full_{col}"] = full_df[col].tolist()

    # JSON metrics
    for col in ["json_exact", "json_key_f1", "json_rougeL", "json_val_cos"]:
        detail_df[col] = json_df[col].tolist()

    detail_df.to_csv(OUT_DETAIL, index=False, encoding="utf-8-sig")

    summary = {
        "full": full_summary,
        "desc": desc_summary,
        "json": json_summary,
    }

    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 50)
    print("             ✨ 분리 평가 결과 저장 완료 ✨             ")
    print("=" * 50)
    print("저장 완료:")
    print(" -", OUT_DETAIL)
    print(" -", OUT_SUMMARY)
    print("\n요약:", json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


✅ (FULL) 108개 샘플에 대해 텍스트 기준 지표 계산

[FULL] 1. ROUGE-L 계산 중 ...


FULL - ROUGE-L:   0%|          | 0/108 [00:00<?, ?it/s]


[FULL] 2. BLEU-4 (sacrebleu) 계산 중 ...


FULL - Per-sample BLEU-4:   0%|          | 0/108 [00:00<?, ?it/s]


[FULL] 3. METEOR 계산 중 (evaluate: meteor) ...


...

JSON - JSON ROUGE-L:   0%|          | 0/108 [00:00<?, ?it/s]


[JSON] JSON Embedding Cosine 계산 중 ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


  ✨ (JSON) JSON 지표 계산 완료 ✨
{
  "json_exact": {
    "mean": 0.0,
    "std": 0.0
  },
  "json_key_f1": {
    "mean": 0.776209,
    "std": 0.387867
  },
  "json_rougeL": {
    "mean": 0.696443,
    "std": 0.307142
  },
  "json_val_cos": {
    "mean": 0.851354,
    "std": 0.232484
  },
  "WeightedScore_json": 0.619119
}

             ✨ 분리 평가 결과 저장 완료 ✨             
저장 완료:
 - ./metrics_detail_split.csv
 - ./metrics_summary_split.json

요약: {
  "full": {
    "rougeL": {
      "mean": 0.731782,
      "std": 0.183154
    },
    "bleu4": {
      "mean": 0.043875,
      "std": 0.040488
    },
    "meteor": {
      "mean": 0.386299,
      "std": 0.151069
    },
    "berts_f1": {
      "mean": 0.87734,
      "std": 0.034009
    },
    "emb_cos": {
      "mean": 0.558782,
      "std": 0.152109
    },
    "bleu4_corpus": 0.037745,
    "WeightedScore": 0.769961
  },
  "desc": {
    "rougeL": {
      "mean": 0.018303,
      "std": 0.109929
    },
    "bleu4": {
      "mean": 0.003194,
      "std": 0.0

In [13]:
import json
import pandas as pd

# 1) JSON 로드
with open("./metrics_summary_split.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

# 2) full/desc/json 각각에서 평균값만 추출
full_metrics = {k: v["mean"] for k, v in summary["full"].items() if isinstance(v, dict)}
desc_metrics = {k: v["mean"] for k, v in summary["desc"].items() if isinstance(v, dict)}
json_metrics = {k: v["mean"] for k, v in summary["json"].items() if isinstance(v, dict)}

# 3) DataFrame으로 정리
df_full = pd.DataFrame(full_metrics, index=["full"])
df_desc = pd.DataFrame(desc_metrics, index=["desc"])
df_json = pd.DataFrame(json_metrics, index=["json"])

df_all = pd.concat([df_full, df_desc, df_json])

print(df_all)

        rougeL     bleu4    meteor  berts_f1   emb_cos  json_exact  \
full  0.731782  0.043875  0.386299  0.877340  0.558782         NaN   
desc  0.018303  0.003194  0.031793  0.857181  0.577475         NaN   
json       NaN       NaN       NaN       NaN       NaN         0.0   

      json_key_f1  json_rougeL  json_val_cos  
full          NaN          NaN           NaN  
desc          NaN          NaN           NaN  
json     0.776209     0.696443      0.851354  
